<a href="https://colab.research.google.com/github/ankitsingh435517/lichess-gpt/blob/main/lichess_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt-get install zstd
!pip install chess

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (510 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directo

In [2]:
import subprocess
import io
import chess.pgn

N_GAMES = 10_000
OUTPUT_FILE = "games.txt"

# Download August 2026 Lichess database and stream-decompress it.
wget = subprocess.Popen(
    [
        "wget",
        "-qO-",
        "https://database.lichess.org/standard/lichess_db_standard_rated_2026-08.pgn.zst",
    ],
    stdout=subprocess.PIPE,
)

zstd = subprocess.Popen(
    ["zstd", "-d"],
    stdin=wget.stdout,
    stdout=subprocess.PIPE,
)

# Allow wget to receive SIGPIPE if zstd exits early.
wget.stdout.close()

# Convert bytes → text so python-chess can parse PGN.
pgn_stream = io.TextIOWrapper(
    zstd.stdout,
    encoding="utf-8"
)

games_written = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

    while games_written < N_GAMES:

        game = chess.pgn.read_game(pgn_stream)

        # End of database
        if game is None:
            break

        # Extract UCI moves
        uci_moves = [
            move.uci()
            for move in game.mainline_moves()
        ]

        # Ignore games with no moves
        if not uci_moves:
            continue

        # One complete game per line
        out.write(" ".join(uci_moves) + "\n")

        games_written += 1

        if games_written % 1000 == 0:
            print(f"Extracted {games_written}/{N_GAMES} games")

pgn_stream.close()

print(f"\nFinished.")
print(f"Games extracted: {games_written}")
print(f"Saved to: {OUTPUT_FILE}")

Extracted 1000/10000 games
Extracted 2000/10000 games
Extracted 3000/10000 games
Extracted 4000/10000 games
Extracted 5000/10000 games
Extracted 6000/10000 games
Extracted 7000/10000 games
Extracted 8000/10000 games
Extracted 9000/10000 games
Extracted 10000/10000 games

Finished.
Games extracted: 10000
Saved to: games.txt


In [3]:
games = []
unique_uci_moves = set()

with open("games.txt", "r", encoding="utf-8") as f:
    for line in f:
        moves = line.strip().split()

        if moves:
            games.append(moves)
            unique_uci_moves.update(moves)

print("Games:", len(games))
print("Unique UCI moves:", len(unique_uci_moves))

Games: 10000
Unique UCI moves: 1858


In [4]:
len(games)

10000

In [5]:
import matplotlib.pyplot as plt


In [6]:
# full move uci tokenizer
vocab = ['.'] + sorted(set(unique_uci_moves))
vocab_size = len(sorted(vocab)) # all the unique moves are the vocabulary
move_to_id = {move:i for i, move in enumerate(vocab)}
id_to_move = {i:move for i, move in enumerate(vocab)}
encode = lambda moves: [move_to_id[move] for move in moves]
decode = lambda ids: ["".join(id_to_move[id] for id in ids)]

print(encode(["e2e4"]))
print(decode(encode(["e2e4"])))

[973]
['e2e4']


In [7]:
import random
random.seed(42)
random.shuffle(games)

n1 = int(0.8 * len(games))
n2 = int(0.9 * len(games))

train_games = games[:n1]
val_games = games[n1:n2]
test_games = games[n2:]

In [8]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
def get_batch_cpu(games, batch_size, context_size, move_to_id):
    X = torch.zeros(
        (batch_size, context_size),
        dtype=torch.long
    )

    Y = torch.zeros(
        batch_size,
        dtype=torch.long
    )

    for i in range(batch_size):
        game = games[torch.randint(len(games), (1,)).item()]
        pos = torch.randint(len(game), (1,)).item()
        context = [0] * context_size

        for move in game[:pos]:
            ix = move_to_id[move]
            context = context[1:] + [ix]

        X[i] = torch.tensor(context)
        Y[i] = move_to_id[game[pos]]

    return X, Y

In [168]:
eval_iters = 200
batch_size= 128
n_blocks = 6
n_embd = 224
n_head = 4
h_dim = 4 * n_embd
context_size = 128

eval_batches = {
    "train": [
        get_batch_cpu(train_games, batch_size, context_size, move_to_id)
        for _ in range(eval_iters)
    ],
    "val": [
        get_batch_cpu(val_games, batch_size, context_size, move_to_id)
        for _ in range(eval_iters)
    ],
}

In [169]:
import torch
import torch.nn as nn
import torch.nn.functional as F
g = torch.Generator().manual_seed(1337)

class MLP(nn.Module):
  def __init__(self, n_embd, h_dim):
    super().__init__()
    self.hidden_layer = nn.Linear(n_embd, h_dim)
    self.relu = nn.ReLU()
    self.output_layer = nn.Linear(h_dim, n_embd)

  def forward(self, x):
    x = self.relu(self.hidden_layer(x))
    out = self.output_layer(x)
    return out

In [170]:
import math

class Head(nn.Module):
  def __init__(self, n_embd, head_size, context_size):
    super().__init__()
    self.head_size = head_size
    self.Q = nn.Linear(n_embd, head_size)
    self.K = nn.Linear(n_embd, head_size)
    self.V = nn.Linear(n_embd, head_size)
    self.register_buffer(
        "tril",
        torch.tril(torch.ones(context_size, context_size)) # to match (B, T, T) each example will have T * T attention matrix where each row is a query and each column of that row is the key it needs to attend
    )

  def forward(self, x):
    q = self.Q(x)
    k = self.K(x)
    v = self.V(x)

    s = q @ k.transpose(-2, -1)
    s = s / math.sqrt(self.head_size)

    # causal mask
    mask = self.tril[:x.size(1), :x.size(1)]
    s = s.masked_fill(mask == 0, float("-inf"))

    w = F.softmax(s, dim=-1)

    o = w @ v

    return o

In [171]:
class MHA(nn.Module):
  def __init__(self, n_embd, head_size, n_head, context_size):
    super().__init__()
    self.heads = nn.ModuleList([
        Head(n_embd, head_size, context_size) for _ in range(n_head)
    ])
    self.out_layer = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    outputs = [head(x) for head in self.heads]
    out = torch.cat(outputs, dim=-1)
    out = self.out_layer(out)
    return out

In [172]:
class Block(nn.Module):
  def __init__(self, n_embd, n_head, h_dim, context_size):
    super().__init__()
    head_size = n_embd // n_head
    self.mha = MHA(n_embd, head_size, n_head, context_size)
    self.mlp = MLP(n_embd, h_dim)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x1 = x + self.mha(self.ln1(x))
    x2 = x1 + self.mlp(self.ln2(x1))
    return x2

In [173]:
class GPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.blocks = nn.ModuleList([
        Block(n_embd, n_head, h_dim, context_size) for _ in range(n_blocks)
    ])
    self.embd_table = nn.Embedding(vocab_size, n_embd)
    self.pos_embd_table = nn.Embedding(context_size, n_embd)
    self.final_ln = nn.LayerNorm(n_embd)
    self.linear = nn.Linear(n_embd, vocab_size)

  def forward(self, x, y=None):
    tok_emb = self.embd_table(x)

    T = tok_emb.shape[1]
    pos_ids = torch.arange(T, device=x.device)
    pos_emb = self.pos_embd_table(pos_ids)

    x = tok_emb + pos_emb

    for block in self.blocks:
      x = block(x)

    x = self.final_ln(x)
    logits = self.linear(x)
    logits = logits[:, -1, :]
    if y is None:
      return logits
    loss = F.cross_entropy(logits, y)
    return logits, loss

In [174]:
# model
model = GPT()

In [175]:
model = model.to(device)

print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [176]:
# optimizer
lr = 3e-4
wd = 0.01

total_params = sum(p.numel() for p in model.parameters())
print(total_params)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

4493955


In [177]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

print("Device:", next(model.parameters()).device)

num_params = sum(p.numel() for p in model.parameters())
model_memory_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / 1024**2

print(f"Model parameters: {num_params / 1e6:.2f}M")
print(f"Model parameter memory: {model_memory_mb:.1f} MB")

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1
Device: cuda:0
Model parameters: 4.49M
Model parameter memory: 17.1 MB
Allocated: 0.17 GB
Reserved:  1.57 GB


In [ ]:
# train
import time
import gc
import torch

# CUDA OOM Related ------------------------------
# Safely delete any older model and optimizer references to release the GPU memory
if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer

# 1. Clean up lingering variables and clear GPU cache to recover the 14+ GB of wasted VRAM
gc.collect()
torch.cuda.empty_cache()

# Now, recreate the model and optimizer fresh in clean memory
from __main__ import GPT, device  # Ensure imports are present
model = GPT().to(device)

# Training code ----------------------------------
lr = 3e-4
wd = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

best_dev_loss = float("inf")
num_steps = 10000

torch.cuda.synchronize()
start = time.time()

for step in range(num_steps):

    # -------------------------
    # Get a random training batch
    # -------------------------
    x, y = get_batch_cpu(
        train_games,
        batch_size,
        context_size,
        move_to_id
    )

    x = x.to(device)
    y = y.to(device)

    # -------------------------
    # Forward pass
    # -------------------------
    logits, loss = model(x, y)
    if step % 1000 == 0:
      print(loss.item())

    # -------------------------
    # Backward pass
    # -------------------------
    # set_to_none=True modestly reduces memory footprint during backward step
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
elapsed = time.time() - start

7.641847610473633


In [ ]:
# Throughput
total_samples = num_steps * batch_size
throughput = total_samples / elapsed

print(f"Training time: {elapsed / 60:.2f} min")
print(f"Time / step: {elapsed / num_steps * 1000:.2f} ms")
print(f"Throughput: {throughput:.1f} samples/sec")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# peak memory
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"Peak:      {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# evaluate train and val los
@torch.no_grad()
def estimate_loss():
    model.eval()

    out = {}

    for split in ["train", "val"]:
        total_loss = 0.0

        for X, Y in eval_batches[split]:
            X = X.to(device)
            Y = Y.to(device)

            _, loss = model(X, Y)
            total_loss += loss.item()

        out[split] = total_loss / len(eval_batches[split])

    model.train()
    return out

In [ ]:
estimate_loss()

In [59]:
def generate(max_new_tokens=50):
  context = [0] * context_size
  out = []
  for _ in range(max_new_tokens):
    # Create the tensor on the exact same device as the model (GPU)
    X = torch.tensor([context], device=device)
    logits = model(X)
    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    tok_id = next_id.item()
    out.append(tok_id)
    context = context[1:] + [tok_id]

    if tok_id == 0:
      break

  return out

generated_games = [decode(generate())[0] for _ in range(50)]
print(generated_games)

['e2e4d6c7g1f3g8f6g1f3b8c6b1d2f8e7b4b5d7d5f1d3e7e6c2c3b8c6e5f7c7c6d3g6d6c5e7d6b1d2b8d7e2d3d7d5d5c3a3a4b4d2f1d3g8f8g1g2g1f1c6d7c4d5g4h5f3e4c6c5a2a3a5b3d7d8e4d5f1f8e8f7e3d2e8f7d7e8e4d5f6g4f5f6b2e5g4g3e5g7', 'g1f3d7d5e4d5d7d5c2c4e7e6h4d8g1f3c8f5c2c3f3f5g8f6f2f4d7c5b7c6f2f4f6d5f1e1d7d6b2b4e7f8f4f5c8e6d2d4e8e5a4c2e6d6e2f4c6d5f1d3c8f5d1d2c6e5f3c3e7c6d1e2d7c5d3g6f8d8e1b1a7a6f7f6c3e4b4c5c7c8f4g3g7c3a2f2d3f5g7c3', 'e2e4c7c5d2d3f8c5h1f1c7c5f1c4c7c6b1c3b8c6a2a3e7e5e1g1f8e7e1g1a8c8h4g3c8e6d1e2d8e7c2d1a5b6d1b3b4c3a3a4e6e5f3g5h1g1b6c5e5h2a3c2h5g4a4c3d1d8g2f1qe6c4h3d3a1d1g8g7h6g7d2f3c8g4c3c4f6f7a2b4e4d6g1h1d7a7f1f6f4g5', 'e2e4d7d5d2d3c7c6b2c3f8d6f2f4f8b4c2c3f8e7f1f7e5f4b5a4d6d7c3e4f8c5f2f3d5c4e1g1e5e4e2e3d7c7f4f5g7f6f1f2f6g6b5a4b8d7d2d1a1d1d8f6a1a3c2c4e7e5c1f4b4c2f3d2h6h4e5g3e7e6g3g6g3f3d7c6e5e6e8g8c2c1e1c1e7g6f4g3g3f3', 'e2e4d7d5f4d2e7e6d2d4d7d5e4d5d5g2d6e5d2b3d7f5a1d1f8g7b1a1c2e4f8e7c3d5c8e6b1d2e5d4c2c4e4g6g5e6d4d3g4e2h3g5c6a5g2g4c7c6d6e4a8c8h3g3f5d3g7h7g3h2c5e4d3d4e4d2d1d8d1c1d3g6f7e6d1f1f8d8d4c3b

In [60]:
import chess

positions = []

for moves in generated_games:
    moves = [moves[i:i+4] for i in range(0, len(moves), 4)]
    board = chess.Board()

    for i, move_uci in enumerate(moves):
        move = chess.Move.from_uci(move_uci)

        if move in board.legal_moves:
            board.push(move)
        else:
            positions.append(i + 1)
            break
    else:
        # Entire generated game was legal
        positions.append(len(moves) + 1)

In [61]:
import numpy as np

positions = np.array(positions)

print("Mean first illegal position:", positions.mean())
print("Median:", np.median(positions))
print("Survive 10 moves:", np.mean(positions > 10))
print("Survive 20 moves:", np.mean(positions > 20))
print("Survive 50 moves:", np.mean(positions > 50))

Mean first illegal position: 3.12
Median: 3.0
Survive 10 moves: 0.0
Survive 20 moves: 0.0
Survive 50 moves: 0.0


In [27]:
# -------
# Start the experiment on mech interp.
# -------
# Write up